Scrape Customer Reviews of Equity Mobile App data from Google Playstore and Apple Store

In [8]:
!python -m pip install google-play-scraper app-store-scraper pandas


   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
   ---------------------------------------- 0/6 [chardet]
  Attempting 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [10]:
from google_play_scraper import reviews, Sort
import pandas as pd

app_id = "ke.co.equitygroup.equitymobile"

result, continuation_token = reviews(
    app_id,
    lang='en',
    country='ke',      # Kenya
    sort=Sort.NEWEST,  # most relevant
    count=2000
)

df_gp = pd.DataFrame(result)
df_gp.to_csv("google_play_equitygroup_reviews.csv", index=False)
df_gp.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,af4c77f2-fecd-48ba-acad-e09a7f220ed8,Hary Zack,https://play-lh.googleusercontent.com/a-/ALV-U...,Amazing,4,0,0.0.392,2026-01-28 12:12:33,NaN,NaT,0.0.392
1,2a2b0c67-b1b5-4041-a521-396d7287b35b,Winnie Gimoro,https://play-lh.googleusercontent.com/a-/ALV-U...,Transactions don't go through,1,0,0.0.392,2026-01-28 12:10:26,NaN,NaT,0.0.392
2,1432df96-460d-4b52-8191-388e3fd1aa30,Mwenda.J Ntongai,https://play-lh.googleusercontent.com/a-/ALV-U...,Wonderful services,5,0,0.0.401,2026-01-28 10:54:46,NaN,NaT,0.0.401
3,c41ab7c0-aa92-4483-ab7a-67a326ff0f94,Mk Aqram pro,https://play-lh.googleusercontent.com/a-/ALV-U...,it's works very fast,5,0,0.0.392,2026-01-28 10:39:28,NaN,NaT,0.0.392
4,377f1838-6853-42d9-ab4b-4d82f731dcef,Chacha Victor,https://play-lh.googleusercontent.com/a/ACg8oc...,excellent,5,0,0.0.392,2026-01-28 10:09:45,NaN,NaT,0.0.392


In [11]:
df_gp = df_gp[['content', 'score', 'at', 'reviewCreatedVersion']]
df_gp.rename(columns={
    'content': 'review',
    'score': 'rating',
    'at': 'date',
    'reviewCreatedVersion': 'app_version'
}, inplace=True)

df_gp.head()

,review,rating,date,app_version
0,Amazing,4,2026-01-28 12:12:33,0.0.392
1,Transactions don't go through,1,2026-01-28 12:10:26,0.0.392
2,Wonderful services,5,2026-01-28 10:54:46,0.0.401
3,it's works very fast,5,2026-01-28 10:39:28,0.0.392
4,excellent,5,2026-01-28 10:09:45,0.0.392


In [14]:
import requests
import pandas as pd
import time

app_id_ios = 1569017982
country_ios = 'ke'

all_reviews_ios = []
page_num = 1
max_reviews = 2000
reviews_per_page = 50 # Typically, the RSS feed returns 50 reviews per page

print(f"Fetching App Store reviews for app_id: {app_id_ios} in country: {country_ios}")

while len(all_reviews_ios) < max_reviews:
    url = f"https://itunes.apple.com/{country_ios}/rss/customerreviews/id={app_id_ios}/sortBy=mostRecent/page={page_num}/json"
    print(f"Requesting page {page_num} from: {url}")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an exception for HTTP errors
        data = response.json()

        # The 'feed' object contains 'entry' which is a list of reviews
        if 'feed' in data and 'entry' in data['feed']:
            entries = data['feed']['entry']
            # The first entry is usually general feed info
            if page_num == 1 and len(entries) > 0 and 'im:rating' not in entries[0]:
                # Skip the first entry if it's not a review
                entries_to_process = entries[1:]
            else:
                entries_to_process = entries

            for entry in entries_to_process:
                review_data = {
                    'review': entry['content']['label'],
                    'rating': int(entry['im:rating']['label']),
                    'date': entry['updated']['label'],
                    'app_version': entry['im:version']['label'] if 'im:version' in entry else None
                }
                all_reviews_ios.append(review_data)

            print(f"Collected {len(all_reviews_ios)} reviews so far.")

            # Check if there are more pages by looking for a 'next' link in the 'link' array
            # This RSS feed typically doesn't have a 'next' link, so we increment page_num and stop if no new reviews
            if len(entries_to_process) < reviews_per_page or len(entries_to_process) == 0:
                print(f"No more reviews found or less than {reviews_per_page} on page {page_num}, stopping.")
                break
            page_num += 1
            time.sleep(1) # Be polite and avoid hitting rate limits
        else:
            print(f"No more entries found in the feed on page {page_num}, stopping.")
            break # No more reviews

    except requests.exceptions.RequestException as e:
        print(f"Request failed for page {page_num}: {e}")
        break # Stop if there's a request error
    except Exception as e:
        print(f"An unexpected error occurred on page {page_num}: {e}")
        break

df_ios = pd.DataFrame(all_reviews_ios)
df_ios.to_csv("apple_store_equitygroup_reviews.csv", index=False)
df_ios.head()

Fetching App Store reviews for app_id: 1569017982 in country: ke
Requesting page 1 from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/page=1/json
Collected 50 reviews so far.
Requesting page 2 from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/page=2/json
Collected 100 reviews so far.
Requesting page 3 from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/page=3/json
Collected 150 reviews so far.
Requesting page 4 from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/page=4/json
Collected 200 reviews so far.
Requesting page 5 from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/page=5/json
Collected 250 reviews so far.
Requesting page 6 from: https://itunes.apple.com/ke/rss/customerreviews/id=1569017982/sortBy=mostRecent/page=6/json
Collected 284 reviews so far.
No more reviews found or less than 50 on page 6, stopping.


,review,rating,date,app_version
0,Good app,5,2026-01-24T06:03:02-07:00,0.4.4
1,Very quick and reliable.,5,2026-01-23T05:09:23-07:00,0.4.4
2,Easy to use,5,2026-01-09T09:33:39-07:00,0.4.3
3,Reliable,5,2026-01-07T04:15:42-07:00,0.4.3
4,I was to be deducted 240 for debit card but go...,5,2025-12-23T23:05:36-07:00,0.4.3


Cleaning the files

In [15]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_gp['clean_review'] = df_gp['review'].apply(clean_text)
df_ios['clean_review'] = df_ios['review'].apply(clean_text)